<a href="https://colab.research.google.com/github/Samikhxnn/xray-disease-identification-customDataset-/blob/main/CustomDataset(disease_detection_from_xRay).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mounting drive to get our data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Importing libraries and modules

In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader


In [ ]:
import torchvision
from torchvision import models
from torchvision import transforms


In [ ]:
from PIL import Image
import os


# Creating Dataset Class

In [ ]:
class CustomMedicalDataset(Dataset):

  def __init__(self,root,transform):
    self.root=root
    self.transform=transform



    self.classes=sorted(os.listdir(root))
    self.class_to_idx={class_name:index  for index,class_name in enumerate(self.classes)}     # classes=["pneumonia","tb","lung_cancer"]



    # create a list of pair of image root and label
    self.images=[]
    for class_name in self.classes:
      class_path=os.path.join(root,class_name)

      for img_name in os.listdir(class_path):
        img_path=os.path.join(class_path,img_name)
        self.images.append(( img_path , self.class_to_idx[class_name] )    )


  def __len__(self):

    return len(self.images)


  def __getitem__(self,index):

    img,label=self.images[index]

    img=Image.open(img).convert("L")


    if self.transform:
      img=self.transform(img)


    return img,label







# transforms

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),  # keeps 3ch for pretrained models
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),                # X-rays can be slightly rotated
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],   # ✅ X-ray specific
                         std=[0.5, 0.5, 0.5])
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5])
])

In [ ]:
train_dataset=CustomMedicalDataset(root="/content/drive/MyDrive/doc/train",transform=transform)
test_dataset=CustomMedicalDataset(root="/content//drive/MyDrive/doc/test",transform=test_transform)

# Dataloader

In [ ]:
train_dataloader=DataLoader(train_dataset,batch_size=2,shuffle=True)
test_dataloader=DataLoader(test_dataset,batch_size=2)

# Get a pretrained model

In [ ]:
model=models.vgg16(pretrained=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
for param in model.features.parameters():
  param.requires_grad=False

# Modifying the clssifier

In [ ]:
model.classifier

Sequential(
  (0): Linear(in_features=25088, out_features=4096, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=4096, out_features=4096, bias=True)
  (4): ReLU(inplace=True)
  (5): Dropout(p=0.5, inplace=False)
  (6): Linear(in_features=4096, out_features=1000, bias=True)
)

In [ ]:
model.classifier=nn.Sequential(
    nn.Linear(25088,1000),
    nn.ReLU(),
    nn.Linear(1000,300),
    nn.ReLU(),
    nn.Linear(300,3)
)

# Set up loss function and Optimizer

In [ ]:
loss_func=nn.CrossEntropyLoss()
o=torch.optim.Adam(
    params=model.classifier.parameters(),
    lr=0.001
)

# training the model

In [ ]:
epochs=10
for epoch in range(epochs):
  t_loss=0
  for x,y in train_dataloader:
    y_logits=model(x)
    loss=loss_func(y_logits,y)
    t_loss+=loss


    o.zero_grad()
    loss.backward()
    o.step()
  print("avg  loss per batch : ",t_loss/len(train_dataloader))

avg  loss per batch :  tensor(1.6431, grad_fn=<DivBackward0>)
avg  loss per batch :  tensor(0.9868, grad_fn=<DivBackward0>)
avg  loss per batch :  tensor(0.7867, grad_fn=<DivBackward0>)
avg  loss per batch :  tensor(0.6336, grad_fn=<DivBackward0>)
avg  loss per batch :  tensor(0.5531, grad_fn=<DivBackward0>)
avg  loss per batch :  tensor(0.2895, grad_fn=<DivBackward0>)
avg  loss per batch :  tensor(0.2409, grad_fn=<DivBackward0>)
avg  loss per batch :  tensor(0.2272, grad_fn=<DivBackward0>)
avg  loss per batch :  tensor(0.4513, grad_fn=<DivBackward0>)
avg  loss per batch :  tensor(0.1491, grad_fn=<DivBackward0>)


# Testing the model

In [ ]:
model.eval()
with torch.no_grad():
  t_acc=0
  for x,y in test_dataloader:
    y_logits=model(x)

    y_prob=torch.softmax(y_logits,dim=1)
    y_class=torch.argmax(y_prob,dim=1)

    acc=(y==y_class).sum()/2
    t_acc+=acc
  print("avg acc per batch :",t_acc/len(test_dataloader))




avg acc per batch : tensor(0.4333)


# accuracy is very bad due to bad data